In [1]:
import spotipy
import os
from spotipy.oauth2 import SpotifyClientCredentials
from spotipy.oauth2 import SpotifyOAuth
from pprint import pprint
import pandas as pd
from dotenv import load_dotenv
import random

In [2]:
'''
dotenv file should contain:
SPOTIPY_CLIENT_ID = "your client id"
SPOTIPY_CLIENT_SECRET = "your client secret"  
SPOTIPY_REDIRECT_URI = "http://localhost:8888/callback"
'''
load_dotenv("C:/apis/.env") # path to your dotenv file
client_id = os.getenv("SPOTIPY_CLIENT_ID")
client_secret = os.getenv("SPOTIPY_CLIENT_SECRET")
redirect_uri = os.getenv("SPOTIPY_REDIRECT_URI")

# This masks your secret keys before printing them, in case you are sharing this notebook:
def mask_secret(unmasked_chars, secret):
    masked_token = secret[:unmasked_chars] + '*' * (len(secret) - unmasked_chars*2) + secret[-unmasked_chars:]
    return masked_token

print(f"SPOTIPY_CLIENT_ID: {mask_secret(4, client_id)}")
print(f"SPOTIPY_CLIENT_SECRET: {mask_secret(4, client_secret)}")
print(f"SPOTIPY_REDIRECT_URI: {redirect_uri}")

SPOTIPY_CLIENT_ID: 3068************************2505
SPOTIPY_CLIENT_SECRET: 9945************************2daa
SPOTIPY_REDIRECT_URI: http://localhost:8888/callback


Checking all my playlists:

In [3]:
#sp = spotipy.Spotify(client_credentials_manager=SpotifyClientCredentials())
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(client_id=os.getenv("SPOTIPY_CLIENT_ID"),
                                               client_secret=os.getenv("SPOTIPY_CLIENT_SECRET"),
                                               redirect_uri=os.getenv("SPOTIPY_REDIRECT_URI"),
                                               scope="playlist-modify-public playlist-modify-private"))

user_id = "duhbeed"
if not user_id:
    raise ValueError("Spotify username is required.")

try:
    results = sp.user_playlists(user_id)
except spotipy.exceptions.SpotifyException as exc:
    print(f"Failed to fetch playlists: {exc}")
else:
    if not results["items"]:
        print("No public playlists found.")
    else:
        while True:
            for playlist in results["items"]:
                print(f"{playlist['name']} ({playlist['tracks']['total']} tracks)")
            if results["next"]:
                results = sp.next(results)
            else:
                break

⌛ Mad Cool 2025 - Jueves (orden horario) ⌛ (100 tracks)
My 2024 Playlist in a Bottle (8 tracks)
Best of 2024 (2/3) | Electronic & Hip-Hop (18 tracks)
Best of 2024 (1/3) | (Mostly) Rock (17 tracks)
Lo Mejor de 2024 (3/3) | de España y/o en castellano (15 tracks)
Dogs of TikTok, YouTube and Instagram (reddgr.com) - sorted by track popularity (43 tracks)
Low Festival 2024  (Sábado) 🌊 ¡Orden horario! ⏰ (100 tracks)
Low Festival 2024 🌊 (Domingo) ¡Orden horario! ⏰ (Domingo) (100 tracks)
Low Festival 2024 🌊 ¡Orden horario! ⏰ (Viernes) (100 tracks)
Talking to Chatbots (Reddgr) (International Playlist) (20 tracks)
Colección de podcasts de Reddgr (17 tracks)
Tomavistas 2024 (orden horario) (146 tracks)
Bands and Artists I've Seen Live (Sorted by artist popularity)  (478 tracks)
Reddgr Curated Podcasts (35 tracks)
Dogs of TikTok, YouTube and Instagram (reddgr.com) (44 tracks)
Talking to Chatbots (TTCB) (15 tracks)
DCODE 2022 by David (112 tracks)
Mad Cool 2022 (Jueves) (121 tracks)
2022 (55 track

Full dataframe with playlist metadata:

In [4]:
playlist_items = []
page = sp.user_playlists(user_id, limit=50)
while page:
    playlist_items.extend(page.get("items", []))
    page = sp.next(page) if page.get("next") else None

column_map = {
    "name": "name",
    "id": "playlist_id",
    # "owner.display_name": "owner_display_name",
    # "owner.id": "owner_id",
    # "public": "public",
    # "collaborative": "collaborative",
    "tracks.total": "tracks_total",
    "description": "description",
    "snapshot_id": "snapshot_id",
    "external_urls.spotify": "external_url",
    "uri": "uri",
    # "primary_color": "primary_color",
}

if playlist_items:
    playlists_raw = pd.json_normalize(playlist_items)
    available_cols = [col for col in column_map if col in playlists_raw.columns]
    playlists_df = (
        playlists_raw[available_cols]
        .rename(columns={col: column_map[col] for col in available_cols})
    )
    playlists_df = playlists_df[[column_map[col] for col in available_cols]]
else:
    playlists_df = pd.DataFrame(columns=list(column_map.values()))

playlists_df.to_pickle("pkl/reddgr_playlists.pkl")
playlists_df.to_csv("csv/reddgr_playlists.csv", index=False)

display(playlists_df)

,name,playlist_id,tracks_total,description,snapshot_id,external_url,uri
0,⌛ Mad Cool 2025 - Jueves (orden horario) ⌛,0acg8XNM0LxaTeSan2U50T,100,100 pistas para preparar el primer día de Mad ...,AAAArqHP41rl5rCp+ib7PEnauUlV+gpj,https://open.spotify.com/playlist/0acg8XNM0Lxa...,spotify:playlist:0acg8XNM0LxaTeSan2U50T
1,My 2024 Playlist in a Bottle,1x5Tv4ITB7rq0OPcUB0NEo,8,A musical time capsule from the past has been ...,AAAAA7aIkhTuCJ9n9Lq8DK1+XwUBCz3o,https://open.spotify.com/playlist/1x5Tv4ITB7rq...,spotify:playlist:1x5Tv4ITB7rq0OPcUB0NEo
2,Best of 2024 (2/3) | Electronic & Hip-Hop,2Bqun7K2S7cAksiUFCQTEm,18,"Every year&#x27;s curated playlists, to be lis...",AAAARHBDRPQyajXV3u4Km8mNKaKRCOoS,https://open.spotify.com/playlist/2Bqun7K2S7cA...,spotify:playlist:2Bqun7K2S7cAksiUFCQTEm
3,Best of 2024 (1/3) | (Mostly) Rock,4AA6jg6T0PbzkVJZKyI5X9,17,"Every year&#x27;s curated playlists, to be lis...",AAAAU5ugERkxM+m+7k0rQJpTmzXlWlaz,https://open.spotify.com/playlist/4AA6jg6T0Pbz...,spotify:playlist:4AA6jg6T0PbzkVJZKyI5X9
4,Lo Mejor de 2024 (3/3) | de España y/o en cast...,1LyCSbuOeqglulGBKP8vSZ,15,Mi sesión de canciones en castellano y de arti...,AAAAVf8WUm9jTLDdOsdAdAumSdFK86qP,https://open.spotify.com/playlist/1LyCSbuOeqgl...,spotify:playlist:1LyCSbuOeqglulGBKP8vSZ
...,...,...,...,...,...,...,...
148,Totally Random Playlist,0AMHQuovdwMQ36DQip0UpQ,20,,AAAAV/jDxNUkR+/zNbkrCdNguzd06UDz,https://open.spotify.com/playlist/0AMHQuovdwMQ...,spotify:playlist:0AMHQuovdwMQ36DQip0UpQ
149,Video Games,0RGDLwNwo4p80AZgms2ZsE,10,,AAAAD9ZjUNOEJc/sSg+Naa7Cv6/oAZoJ,https://open.spotify.com/playlist/0RGDLwNwo4p8...,spotify:playlist:0RGDLwNwo4p80AZgms2ZsE
150,Violence,6AUxha1PnIg6mvvklt0j6n,11,,AAAAD75grCTf+eBiHcDpAZ/bJTKw9g26,https://open.spotify.com/playlist/6AUxha1PnIg6...,spotify:playlist:6AUxha1PnIg6mvvklt0j6n
151,Weezer 10,0erlbFfIKEV4eMJEtKhuoV,10,,AAAAHSKE24JOgaiS2tjbSwNI3FtEm/sr,https://open.spotify.com/playlist/0erlbFfIKEV4...,spotify:playlist:0erlbFfIKEV4eMJEtKhuoV


In [8]:
playlists_df.sample(5)

,name,playlist_id,tracks_total,description,snapshot_id,external_url,uri
10,Colección de podcasts de Reddgr,6JKM7yKOA0RIvxtK0iZ4V1,17,En esta playlist comparto algunos de los podca...,AAAAIixrpzgfTFCndYKvbaraqeevWfk2,https://open.spotify.com/playlist/6JKM7yKOA0RI...,spotify:playlist:6JKM7yKOA0RIvxtK0iZ4V1
109,1985,5x8JXqhNJhJdbDwUXXUgFE,23,,AAAAPXR7ftY16J5GITf5DRoXShHAjj9U,https://open.spotify.com/playlist/5x8JXqhNJhJd...,spotify:playlist:5x8JXqhNJhJdbDwUXXUgFE
55,Best of 2017 (3/6) (Alt. Electronic),0mbkLIeePwUqBuaSM8nHUB,17,,AAAAds83W6HnkI59CDDQS23sIC1TQE3p,https://open.spotify.com/playlist/0mbkLIeePwUq...,spotify:playlist:0mbkLIeePwUqBuaSM8nHUB
38,Best of 2019 (2/6),2r4zhh6Pb3TrFuvNIkoJuG,14,"My 100-song selection of every year, split int...",AAAArcvwHormMpkM+/PRgTUNglY675fM,https://open.spotify.com/playlist/2r4zhh6Pb3Tr...,spotify:playlist:2r4zhh6Pb3TrFuvNIkoJuG
12,Bands and Artists I've Seen Live (Sorted by ar...,5ZAVOxwjVynsdh4FXypiI7,478,"For over 20 years, I&#x27;ve been logging the ...",AAAAJu5kWB7R3ZBwVTkh3gmXBGMp+Rt6,https://open.spotify.com/playlist/5ZAVOxwjVyns...,spotify:playlist:5ZAVOxwjVynsdh4FXypiI7


In [11]:
bestof_year_playlists_df = (
    playlists_df[playlists_df["name"].str.lower().str.startswith(("best of", "lo mejor de"))]
    .reset_index(drop=True)
)

bestof_year_playlists_df = bestof_year_playlists_df.assign(
    year=pd.to_numeric(
        bestof_year_playlists_df["name"].str.extract(r"\b(2[0-9]{3})\b", expand=False),
        errors="coerce"
    )
)
main_cols = ["name", "year", "tracks_total", "description"]
cols = main_cols + [col for col in bestof_year_playlists_df.columns if col not in main_cols]
bestof_year_playlists_df = bestof_year_playlists_df[cols]

bestof_year_playlists_df = bestof_year_playlists_df.sort_values("year", ascending=True, na_position="last")
display(bestof_year_playlists_df)

,name,year,tracks_total,description,playlist_id,snapshot_id,external_url,uri
19,Best of 2010,2010,102,,5iBmxKUnNKzEZoYXxHYC2r,AAABUvdiqXREh7R4u2TxzLIgINVccq2z,https://open.spotify.com/playlist/5iBmxKUnNKzE...,spotify:playlist:5iBmxKUnNKzEZoYXxHYC2r
20,Best of 2011,2011,101,,2lMwz55DQSDpnNed34XFRa,AAACCO8tlBBDSkBWi+zh+FUIVowCy5pz,https://open.spotify.com/playlist/2lMwz55DQSDp...,spotify:playlist:2lMwz55DQSDpnNed34XFRa
71,Best of 2012 (bonus tracks),2012,23,,4jVzmc44I1IfrvgB2gaCIV,AAABztHil5w7vv0K1kpcb/CTmBbaf6u/,https://open.spotify.com/playlist/4jVzmc44I1If...,spotify:playlist:4jVzmc44I1IfrvgB2gaCIV
69,Best of 2012 (4/5) (electronic/hip-hop),2012,20,,73kk51cnXJvlsLp2LOi6wD,AAAAQvlfSNVRg/p7A6FQ20ae2iLiYYWA,https://open.spotify.com/playlist/73kk51cnXJvl...,spotify:playlist:73kk51cnXJvlsLp2LOi6wD
68,Best of 2012 (3/5) (experimental/other),2012,20,,4KYIK7rzuEO96twgvOynvM,AAAAVUe3z/AAcc+u3FJm1qbZaOKUUM1F,https://open.spotify.com/playlist/4KYIK7rzuEO9...,spotify:playlist:4KYIK7rzuEO96twgvOynvM
...,...,...,...,...,...,...,...,...
4,Best of 2023 (2/3),2023,18,"In this year&#x27;s collection, I&#x27;ve sele...",4XOMosM5oObvrcQVFCamFj,AAAAN2HQJgGJ1y7a9q0OzXqWJ9a+kB5H,https://open.spotify.com/playlist/4XOMosM5oObv...,spotify:playlist:4XOMosM5oObvrcQVFCamFj
3,Best of 2023 (1/3),2023,18,"In this year&#x27;s collection, I&#x27;ve sele...",34wg9M1ElBgos2l6QGExCd,AAAAPCLp2VtHkE0rm+UTkhq9PEIEf1zL,https://open.spotify.com/playlist/34wg9M1ElBgo...,spotify:playlist:34wg9M1ElBgos2l6QGExCd
2,Lo Mejor de 2024 (3/3) | de España y/o en cast...,2024,15,Mi sesión de canciones en castellano y de arti...,1LyCSbuOeqglulGBKP8vSZ,AAAAVf8WUm9jTLDdOsdAdAumSdFK86qP,https://open.spotify.com/playlist/1LyCSbuOeqgl...,spotify:playlist:1LyCSbuOeqglulGBKP8vSZ
1,Best of 2024 (1/3) | (Mostly) Rock,2024,17,"Every year&#x27;s curated playlists, to be lis...",4AA6jg6T0PbzkVJZKyI5X9,AAAAU5ugERkxM+m+7k0rQJpTmzXlWlaz,https://open.spotify.com/playlist/4AA6jg6T0Pbz...,spotify:playlist:4AA6jg6T0PbzkVJZKyI5X9


In [15]:
target_playlist = bestof_year_playlists_df.iloc[random.randint(0, len(bestof_year_playlists_df) - 1)]

def format_duration(ms):
    if ms is None:
        return None
    minutes, seconds = divmod(int(ms) // 1000, 60)
    return f"{minutes}:{seconds:02d}"

tracks_data = []
offset = 0
position = 1
while True:
    response = sp.playlist_items(target_playlist["playlist_id"], offset=offset, limit=100)
    items = response.get("items", [])
    if not items:
        break
    for item in items:
        track = item.get("track")
        if not track or track.get("type") != "track":
            continue
        album = track.get("album") or {}
        tracks_data.append({
            "position": position,
            "artists": ", ".join(artist.get("name") for artist in track.get("artists", [])),
            "track_name": track.get("name"),
            "album_name": album.get("name"),
            "release_date": album.get("release_date"),
            "track_popularity": track.get("popularity"),
            "duration_ms": track.get("duration_ms"),
            "added_at": item.get("added_at"),         
            "duration_mmss": format_duration(track.get("duration_ms")),           
            "external_url": (track.get("external_urls") or {}).get("spotify"),
            "track_id": track.get("id"),
            "track_uri": track.get("uri"),
        })
        position += 1
    offset += len(items)
    if not response.get("next"):
        break

if tracks_data:
    playlist_tracks_df = pd.DataFrame(tracks_data)
    playlist_tracks_df["release_date"] = pd.to_datetime(playlist_tracks_df["release_date"], errors="coerce")
    playlist_tracks_df["added_at"] = pd.to_datetime(playlist_tracks_df["added_at"], errors="coerce")
else:
    playlist_tracks_df = pd.DataFrame(columns=[
        "position", "track_id", "track_uri", "track_name", "artists", "album_name",
        "release_date", "added_at", "duration_ms", "duration_mmss", "track_popularity", "external_url"
    ])

display(target_playlist[["name", "playlist_id", "tracks_total"]].to_frame().T)
display(playlist_tracks_df)

,name,playlist_id,tracks_total
21,Best of 2019 (1/6),2HI38LDbMZ9kWwbIqwqFwr,18


,position,artists,track_name,album_name,release_date,track_popularity,duration_ms,added_at,duration_mmss,external_url,track_id,track_uri
0,1,Cage The Elephant,Ready To Let Go,Social Cues,2019-04-19,61,187973,2019-10-31 16:22:45+00:00,3:07,https://open.spotify.com/track/4UyAtnwhaKv7EG1...,4UyAtnwhaKv7EG1BdkBYRI,spotify:track:4UyAtnwhaKv7EG1BdkBYRI
1,2,Vampire Weekend,This Life,Father of the Bride,2019-05-03,59,268600,2019-10-31 16:47:01+00:00,4:28,https://open.spotify.com/track/4dRqYKhLVujxiBX...,4dRqYKhLVujxiBXcq50YzG,spotify:track:4dRqYKhLVujxiBXcq50YzG
2,3,Brittany Howard,Stay High,Jaime,2019-09-20,0,191395,2019-12-02 11:52:52+00:00,3:11,https://open.spotify.com/track/3sjew5VKI4UlxZF...,3sjew5VKI4UlxZF7r8lpMd,spotify:track:3sjew5VKI4UlxZF7r8lpMd
3,4,Michael Kiwanuka,Hero,KIWANUKA,2019-10-10,53,199133,2019-11-26 10:32:28+00:00,3:19,https://open.spotify.com/track/2Y61ZvCuuLOhdkg...,2Y61ZvCuuLOhdkgl1B9ekh,spotify:track:2Y61ZvCuuLOhdkgl1B9ekh
4,5,The Black Keys,Every Little Thing,"""Let's Rock""",2019-06-28,40,199783,2019-12-02 11:06:27+00:00,3:19,https://open.spotify.com/track/4l8f1sX5rJyUphT...,4l8f1sX5rJyUphT1PfG98v,spotify:track:4l8f1sX5rJyUphT1PfG98v
5,6,"Khruangbin, Leon Bridges",Texas Sun,Texas Sun,2019-12-04,0,252811,2019-12-20 17:32:26+00:00,4:12,https://open.spotify.com/track/3k5oLgungD1dSOG...,3k5oLgungD1dSOGLqQdIQw,spotify:track:3k5oLgungD1dSOGLqQdIQw
6,7,Cass McCombs,The Great Pixley Train Robbery,Tip of the Sphere,2019-02-08,19,240821,2019-10-31 16:37:26+00:00,4:00,https://open.spotify.com/track/4oRzacJVFt3KiXe...,4oRzacJVFt3KiXehJmRyXb,spotify:track:4oRzacJVFt3KiXehJmRyXb
7,8,Iggy Pop,James Bond,James Bond,2019-07-30,23,271171,2019-10-31 16:39:04+00:00,4:31,https://open.spotify.com/track/2iStSsIolUcFI9E...,2iStSsIolUcFI9Eb3cVWnk,spotify:track:2iStSsIolUcFI9Eb3cVWnk
8,9,"Perry Farrell, Etty Lau Farrell",Machine Girl (feat. Etty Lau Farrell),Kind Heaven,2019-06-07,0,215586,2019-10-31 18:35:31+00:00,3:35,https://open.spotify.com/track/5EhpUXtNgSraMRs...,5EhpUXtNgSraMRs8nFHZ8i,spotify:track:5EhpUXtNgSraMRs8nFHZ8i
9,10,Pixies,Catfish Kate,Beneath the Eyrie,2019-09-13,0,188306,2019-11-02 14:40:58+00:00,3:08,https://open.spotify.com/track/30BD3DBxv3wnyBx...,30BD3DBxv3wnyBxTinDeVJ,spotify:track:30BD3DBxv3wnyBxTinDeVJ
